# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aamnamalik16-bit/flyrank-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

if not os.path.exists('flyrank-ML-internship'):
    !git clone https://github.com/aamnamalik16-bit/flyrank-ML-internship.git

os.chdir('flyrank-ML-internship')

!pip install duckdb -q

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
print("Setup complete!")

Cloning into 'flyrank-ML-internship'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 168 (delta 76), reused 111 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (168/168), 1.86 MiB | 8.10 MiB/s, done.
Resolving deltas: 100% (76/76), done.
Setup complete!


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule in plain words:

A query-page pair is worth reviewing if it was getting impressions recently but its clicks are dropping compared to the previous period. This suggests the page is visible for the query but failing to capture clicks — a sign the title, meta, or content may need attention.

Two signals I am checking:

impressions_last30 — is the page still visible for this query? (volume signal)
click_delta = clicks_last30 - clicks_prev30 — is click momentum dropping? (trend signal)

One reason code: visible_but_losing_clicks — page has impressions but negative click delta.

Action label: review_query_targeting

In [2]:
# Signal check 1 — impressions distribution
df = con.sql(f"""
    SELECT
        CASE
            WHEN impressions_last30 = 0 THEN 'no_impressions'
            WHEN impressions_last30 < 10 THEN 'low_1_to_9'
            WHEN impressions_last30 < 100 THEN 'medium_10_to_99'
            ELSE 'high_100_plus'
        END as impression_bucket,
        COUNT(*) as n
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
    GROUP BY impression_bucket
    ORDER BY n DESC
""").df()

print("Signal 1: Impressions distribution")
print("VERDICT: CONFIRMED — most rows have low impressions; high-impression pairs are rare and valuable")
print(df.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1: Impressions distribution
VERDICT: CONFIRMED — most rows have low impressions; high-impression pairs are rare and valuable
impression_bucket       n
       low_1_to_9 1044833
  medium_10_to_99  735688
   no_impressions  530758
    high_100_plus  102969


In [3]:
# Signal check 2 — click delta distribution
df2 = con.sql(f"""
    SELECT
        CASE
            WHEN (clicks_last30 - clicks_prev30) < 0 THEN 'losing_clicks'
            WHEN (clicks_last30 - clicks_prev30) = 0 THEN 'no_change'
            WHEN (clicks_last30 - clicks_prev30) > 0 THEN 'gaining_clicks'
        END as click_trend,
        COUNT(*) as n
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
    WHERE (impressions_last30 > 0) IS TRUE
    GROUP BY click_trend
    ORDER BY n DESC
""").df()

print("Signal 2: Click delta distribution (among rows with impressions)")
print("VERDICT: CONFIRMED — losing_clicks pairs are common; these are our targets")
print(df2.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 2: Click delta distribution (among rows with impressions)
VERDICT: CONFIRMED — losing_clicks pairs are common; these are our targets
   click_trend       n
     no_change 1746201
 losing_clicks   72544
gaining_clicks   64745


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# Build the baseline score and ranked queue
baseline_df = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        query_hash_id,
        impressions_last30,
        clicks_last30,
        clicks_prev30,
        avg_position_last30,
        (clicks_last30 - clicks_prev30) as click_delta,
        -- BASELINE SCORE: visible but losing clicks
        CASE
            WHEN (clicks_last30 - clicks_prev30) < 0
             AND impressions_last30 >= 10
            THEN impressions_last30 * ABS(clicks_last30 - clicks_prev30)
            ELSE 0
        END as baseline_score,
        -- REASON CODE
        CASE
            WHEN (clicks_last30 - clicks_prev30) < 0
             AND impressions_last30 >= 10
            THEN 'visible_but_losing_clicks'
            ELSE 'no_action'
        END as reason_code,
        -- ACTION LABEL
        CASE
            WHEN (clicks_last30 - clicks_prev30) < 0
             AND impressions_last30 >= 10
            THEN 'review_query_targeting'
            ELSE 'monitor'
        END as action_label
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
    WHERE (impressions_last30 > 0) IS TRUE
    ORDER BY baseline_score DESC
""").df()

# Write the CSV
import os
os.makedirs('work/outputs', exist_ok=True)
baseline_df.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Total rows: {len(baseline_df)}")
print(f"Flagged for review: {(baseline_df['reason_code'] == 'visible_but_losing_clicks').sum()}")
print("CSV written to work/outputs/baseline_action_score.csv")
baseline_df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows: 1883490
Flagged for review: 55340
CSV written to work/outputs/baseline_action_score.csv


,content_hash_id,client_hash_id,query_hash_id,impressions_last30,clicks_last30,clicks_prev30,avg_position_last30,click_delta,baseline_score,reason_code,action_label
0,content_987d251ee617d9c6,client_73cda7b4e4f265ea,query_ab81171134a428bb,56140,99,281,3.562380,-182,10217480,visible_but_losing_clicks,review_query_targeting
1,content_df47d1b976106de4,client_23a62021009f63c4,query_6dbfdad311299b16,34736,39,116,11.391784,-77,2674672,visible_but_losing_clicks,review_query_targeting
2,content_471d9cabce329a66,client_73cda7b4e4f265ea,query_9c9cf9ca4413e277,42074,19,72,8.373770,-53,2229922,visible_but_losing_clicks,review_query_targeting
3,content_943dc881428182b8,client_8ddc46da5414ffd8,query_1e12d78d0219e482,233451,23,28,1.238611,-5,1167255,visible_but_losing_clicks,review_query_targeting
4,content_145680ddd5f91ea9,client_73cda7b4e4f265ea,query_b9372bdd1f1ba921,18613,37,96,1.832268,-59,1098167,visible_but_losing_clicks,review_query_targeting
5,content_36e53e9c707674fc,client_23a62021009f63c4,query_e01dcb4374107487,32674,53,84,25.357287,-31,1012894,visible_but_losing_clicks,review_query_targeting
6,content_c2d2e725a72e8c25,client_73cda7b4e4f265ea,query_ecd9aaef6f4a2e94,15791,40,101,2.431512,-61,963251,visible_but_losing_clicks,review_query_targeting
7,content_b556f0bd87d6fcca,client_73cda7b4e4f265ea,query_c3575a388b4c0dce,23050,21,53,5.173102,-32,737600,visible_but_losing_clicks,review_query_targeting
8,content_0c99955eba3ea776,client_73cda7b4e4f265ea,query_5dc861ca48ee9162,17003,17,57,4.763454,-40,680120,visible_but_losing_clicks,review_query_targeting
9,content_d0b39ce8464330e6,client_73cda7b4e4f265ea,query_3b8f5d0b6cd8ad6e,37288,30,47,7.132643,-17,633896,visible_but_losing_clicks,review_query_targeting


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
# Show top 10 for review
top10 = baseline_df[baseline_df['reason_code'] == 'visible_but_losing_clicks'].head(10)
print("Top 10 flagged rows:")
display(top10[['content_hash_id', 'impressions_last30', 'clicks_last30',
               'clicks_prev30', 'click_delta', 'avg_position_last30',
               'baseline_score', 'reason_code', 'action_label']])

Top 10 flagged rows:


,content_hash_id,impressions_last30,clicks_last30,clicks_prev30,click_delta,avg_position_last30,baseline_score,reason_code,action_label
0,content_987d251ee617d9c6,56140,99,281,-182,3.562380,10217480,visible_but_losing_clicks,review_query_targeting
1,content_df47d1b976106de4,34736,39,116,-77,11.391784,2674672,visible_but_losing_clicks,review_query_targeting
2,content_471d9cabce329a66,42074,19,72,-53,8.373770,2229922,visible_but_losing_clicks,review_query_targeting
3,content_943dc881428182b8,233451,23,28,-5,1.238611,1167255,visible_but_losing_clicks,review_query_targeting
4,content_145680ddd5f91ea9,18613,37,96,-59,1.832268,1098167,visible_but_losing_clicks,review_query_targeting
5,content_36e53e9c707674fc,32674,53,84,-31,25.357287,1012894,visible_but_losing_clicks,review_query_targeting
6,content_c2d2e725a72e8c25,15791,40,101,-61,2.431512,963251,visible_but_losing_clicks,review_query_targeting
7,content_b556f0bd87d6fcca,23050,21,53,-32,5.173102,737600,visible_but_losing_clicks,review_query_targeting
8,content_0c99955eba3ea776,17003,17,57,-40,4.763454,680120,visible_but_losing_clicks,review_query_targeting
9,content_d0b39ce8464330e6,37288,30,47,-17,7.132643,633896,visible_but_losing_clicks,review_query_targeting


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
print("Leakage check:")
print("- No product flags used")
print("- No future window columns used")
print("- click_delta computed from past windows only")
print("- LEAKAGE CHECK: PASSED")

Leakage check:
- No product flags used
- No future window columns used
- click_delta computed from past windows only
- LEAKAGE CHECK: PASSED


Weak pick: Row 3 has 233,451 impressions but only a 5-click drop — the score is inflated by volume, not meaningful decline. Fix: add a minimum ABS(click_delta) >= 10 filter.

Leakage check:

No product flags used — reason code is computed from raw impressions and clicks only.
No future window used — both clicks_last30 and clicks_prev30 are already observed at decision time.
click_delta is computed from past windows only — no leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.